## Séries : 

- Créer une table `series` 
    - Avec une colonne d'id (comme ci-dessous) : `id`
        - `INTEGER PRIMARY KEY AUTOINCREMENT`
    - Le nom de la série : `name` ou `title`
        - TEXT NOT NULL
    - Le nom du réalisateur : `director`
        - TEXT NOT NULL
    - La date de sortie (date du premier épisode de la saison 1) : `air_date` ou `release_date`
        - Faire une petite recherche pour voir comment stocker une date
        - Décider si vous voulez qu'elle puisse être nulle ou non
    - La note de la série (IMDB / Senscritique / Allociné) : `rating`
        - REAL (vous pouvez faire une petite recherche pour trouver un type précis)
     
- Insérer les données de 2-5 séries
    - Wikipedia en one shot très probable
 
- Afficher toutes les lignes de la table

- 1- Ajouter une colonne pour la date du dernier épisode (en utilisant une clause d'UPDATE)
- 2- Créer une colonne qui contient le nombre de jours entre le premier épisode et le dernier (avec SQL, ne pas les compter à la main)

In [1]:
import sqlite3
from pprint import pprint

conn = sqlite3.connect("series.db")
cur = conn.cursor()

cur.execute("SELECT sqlite_version()")
print(cur.fetchone())

('3.51.0',)


In [2]:
cur.execute("""
CREATE TABLE IF NOT EXISTS series (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    director TEXT NOT NULL,
    release_date  TEXT NOT NULL,
    rating REAL
)
""")

cur.executemany(
    "INSERT INTO series (name, director, release_date, rating) VALUES (?, ?, ?, ?)",
    [("Breaking Bad", "Vince Gilligan", "2008-01-20", 9.5), ("Chernobyl", "Johan Renck", "2019-05-06", 9.4),
    ("Stranger Things", "The Duffer Brothers", "2016-07-15", 8.7)],
)
conn.commit()




In [3]:
for row in cur.execute("SELECT * FROM series"):
    pprint(row)

(1, 'Breaking Bad', 'Vince Gilligan', '2008-01-20', 9.5)
(2, 'Chernobyl', 'Johan Renck', '2019-05-06', 9.4)
(3, 'Stranger Things', 'The Duffer Brothers', '2016-07-15', 8.7)


In [4]:
conn.close()

# 1. AJOUT LA DATE DE FIN

In [5]:
# On rouvre la connexion si elle était fermée
conn = sqlite3.connect("series.db")
cur = conn.cursor()

In [6]:
cur.execute("ALTER TABLE series ADD COLUMN end_date TEXT")
conn.commit()
print("Colonne end_date ajoutée avec succès !")

Colonne end_date ajoutée avec succès !


In [7]:
cur.executemany(
    "UPDATE series SET end_date = ? WHERE name = ?", 
    [("2013-09-29", "Breaking Bad"),
    ("2019-06-03", "Chernobyl"),
    ("2022-07-01", "Stranger Things")]
)
conn.commit()

In [8]:
for row in cur.execute("SELECT * FROM series"):
    pprint(row)

(1, 'Breaking Bad', 'Vince Gilligan', '2008-01-20', 9.5, '2013-09-29')
(2, 'Chernobyl', 'Johan Renck', '2019-05-06', 9.4, '2019-06-03')
(3, 'Stranger Things', 'The Duffer Brothers', '2016-07-15', 8.7, '2022-07-01')


# Créer une colonne qui contient le nombre de jours entre le premier épisode et le dernier

In [9]:
#Création de la colonne pour stocker le nombre de jours (INTEGER)

cur.execute("ALTER TABLE series ADD COLUMN duration_days INTEGER")
print("Colonne 'duration_days' ajoutée avec succès !")

# 2. Calcul SQL de la différence entre les deux dates
# julianday() transforme les dates "YYYY-MM-DD" en nombre de jours pour pouvoir faire la soustraction
cur.execute("""
UPDATE series 
SET duration_days = CAST(julianday(end_date) - julianday(release_date) AS INTEGER)
""")

# On valide les changements
conn.commit()
print("Calcul de la durée effectué pour toutes les séries !")

Colonne 'duration_days' ajoutée avec succès !
Calcul de la durée effectué pour toutes les séries !


In [10]:
#Pour vérifier le résultat :
cur.execute("SELECT name, release_date, end_date, duration_days FROM series ORDER BY duration_days")
for row in cur.fetchall():
    print(f"Série : {row[0]} | Durée : {row[3]} jours")

Série : Chernobyl | Durée : 28 jours
Série : Breaking Bad | Durée : 2079 jours
Série : Stranger Things | Durée : 2177 jours


In [11]:
conn.close()